# Analisi Modale Acustica con pyCAFE
## Guida pratica per principianti

---

Questo notebook ti guida passo dopo passo nell'eseguire un'**analisi modale acustica** usando il software pyCAFE.

Non preoccuparti se non conosci l'acustica: ogni passaggio è spiegato in modo semplice.

---

### Cos'è l'analisi modale acustica?

Immagina di pizzicare una corda di chitarra: vibra a frequenze ben precise, produce un suono caratteristico. Lo stesso fenomeno esiste per l'**aria all'interno di una stanza**: esistono frequenze particolari a cui la stanza "risuona". Queste frequenze si chiamano **frequenze proprie** (o naturali), e le corrispondenti distribuzioni di pressione si chiamano **modi propri** (o modi acustici).

**Perché è utile studiarli?**
- Se una sorgente sonora (altoparlante, strumento musicale) emette suono a una frequenza vicina a un modo proprio, il suono viene molto amplificato → **risonanza**.
- I modi propri determinano come il suono si distribuisce nello spazio.
- Sono fondamentali nella progettazione di sale da concerto, studi di registrazione, abitacoli di automobili, ecc.

Con questo notebook puoi:
1. Caricare una geometria (mesh) rettangolare 2D
2. Scegliere le condizioni al contorno su ogni bordo
3. Calcolare i modi propri e le frequenze proprie
4. Visualizzare la distribuzione di pressione per ogni modo


---
## Passo 1 — Importa pyCAFE

Prima di tutto importiamo la libreria `pycafe`. Questo rende disponibili tutti gli strumenti del software.

In [ ]:
import pycafe
import numpy as np
import matplotlib.pyplot as plt

# Importa il solver modale e le funzioni di post-processing
from pycafe.solver.solver_modale import solve_modal_acoustic_reduced
from pycafe.build_matrices.assembly_cquad8 import expand_mode_to_full

print("pyCAFE importato correttamente!")

---
## Passo 2 — Scegli il fluido

L'analisi acustica dipende dalle proprietà del fluido che riempie il dominio.
Le proprietà fondamentali sono:
- **densità** (`rho`): massa per unità di volume [kg/m³]
- **velocità del suono** (`c0`): quanto velocemente si propaga il suono nel fluido [m/s]

pyCAFE ha due fluidi predefiniti:

| Fluido | Densità [kg/m³] | Velocità del suono [m/s] |
|--------|-----------------|---------------------------|
| `"air"` (aria) | 1.204 | 343 |
| `"water"` (acqua) | 998 | 1480 |

> **Nota:** Le frequenze proprie dipendono direttamente da `c0`. Se usi l'acqua invece dell'aria, le frequenze saranno circa 4× più alte (dato che c_acqua ≈ 4 × c_aria).

In [ ]:
# ================================================================
# SCEGLI IL FLUIDO
# Opzioni disponibili: "air" oppure "water"
# ================================================================
FLUIDO = "air"

fluid = pycafe.load_fluid(FLUIDO)

print(f"Fluido selezionato : {fluid['name']}")
print(f"Densita'           : {fluid['rho']} kg/m^3")
print(f"Velocita' del suono: {fluid['c0']} m/s")
print(f"Impedenza car.     : {fluid['Z0']:.1f} Pa*s/m")

---
## Passo 3 — Carica la mesh

La **mesh** è la discretizzazione del dominio geometrico in tanti piccoli elementi (quadrilateri, in questo caso). Il metodo degli elementi finiti (FEM) risolve il problema dell'acustica su questa griglia discreta.

Il file `.msh` contiene:
- le coordinate di tutti i **nodi** (punti della griglia)
- la connettività degli **elementi** (quali nodi formano ogni elemento)
- i **bordi** del dominio con i loro nomi

> **Cosa vedrai dopo:** pyCAFE stampa i nomi dei bordi disponibili e mostra la mesh graficamente. **Prendi nota dei nomi dei bordi**: ti serviranno nel Passo 4.

In [ ]:
# ================================================================
# CARICA LA MESH
# Assicurati che il file .msh si trovi nella stessa cartella
# di questo notebook, oppure specifica il percorso completo.
# ================================================================
FILE_MESH = "rectangle_CQUAD8.msh"

nodes, elements, boundaries = pycafe.load_mesh(FILE_MESH)

print(f"\nNumero di nodi   : {nodes.shape[0]}")
print(f"Numero di bordi  : {len(boundaries)}")
print("\nBordi disponibili (copiali per il Passo 4!):")
for name in boundaries:
    print(f"   '{name}'")

---
## Passo 4 — Condizioni al contorno

Le **condizioni al contorno** (CC) dicono al software cosa succede fisicamente ai bordi del dominio.
In acustica, le due condizioni più comuni sono:

---

### `"hard_wall"` — Parete rigida (condizione di Neumann)

La parete è **perfettamente rigida e riflettente**: il fluido non può attraversarla, quindi la velocità normale alla parete è zero.

Matematicamente: `∂p/∂n = 0`

**Effetto fisico:** la pressione può assumere valori qualsiasi alla parete, inclusi i massimi (antinodi). È la condizione tipica di pareti solide, pavimento, soffitto.

---

### `"pressure_zero"` — Pressione nulla (condizione di Dirichlet)

La pressione acustica al bordo è **forzata a zero**.

Matematicamente: `p = 0`

**Effetto fisico:** corrisponde a un bordo **aperto** verso l'esterno (es. una finestra completamente aperta su un ambiente infinito). Il bordo assorbe completamente le onde: le onde non si riflettono ma ci sono nodi di pressione al bordo.

---

### Come cambia la fisica?

| Configurazione | Effetto sui modi |
|---|---|
| Tutti `hard_wall` | Stanza completamente chiusa. Il modo (0,0) ha f=0 e viene scartato. I modi iniziano dal (1,0) e (0,1). La pressione ha massimi ai bordi. |
| Tutti `pressure_zero` | Stanza completamente aperta (ideale). Tutti i bordi sono nodi di pressione. |
| Mix | Comportamento intermedio, modi asimmetrici. |

> **Formula indicativa per una stanza rettangolare** (solo pareti rigide):
> 
> `f(m,n) = (c0/2) * sqrt( (m/Lx)^2 + (n/Ly)^2 )`
>
> dove `m, n = 0, 1, 2, ...` sono gli indici di modo, `Lx` e `Ly` le dimensioni.

In [ ]:
# ================================================================
# CONFIGURA LE CONDIZIONI AL CONTORNO
#
# Istruzioni:
#   1. Guarda i nomi dei bordi stampati nel Passo 3
#   2. Scrivi i nomi esatti come chiavi del dizionario
#   3. Per ogni bordo scegli:
#         "hard_wall"     → parete rigida (riflettente)
#         "pressure_zero" → pressione nulla (bordo aperto)
#
# Esempio: una stanza chiusa su tre lati e aperta sul quarto:
#   "bottom": "hard_wall"
#   "top":    "pressure_zero"
#   "left":   "hard_wall"
#   "right":  "hard_wall"
# ================================================================

bc_config = {
    "bottom": "hard_wall",      # <-- modifica qui
    "top":    "hard_wall",      # <-- modifica qui
    "left":   "hard_wall",      # <-- modifica qui
    "right":  "hard_wall",      # <-- modifica qui
}

# --- Controllo: i nomi devono esistere nella mesh ---
for name in bc_config:
    if name not in boundaries:
        print(f"ATTENZIONE: il bordo '{name}' non esiste nella mesh!")
        print(f"Bordi disponibili: {list(boundaries.keys())}")

print("Condizioni al contorno configurate:")
for name, bc_type in bc_config.items():
    label = "Parete rigida" if bc_type == "hard_wall" else "Pressione = 0"
    print(f"   {name:15s} -> {label}")

In [ ]:
# ================================================================
# Conversione del dizionario nel formato interno di pyCAFE
# (non modificare questa cella)
# ================================================================

def build_bc_from_config(bc_config):
    """
    Converte il dizionario bc_config nel formato richiesto da pyCAFE.
    Per l'analisi modale solo la lista dei bordi a pressione zero e'
    rilevante; tutti gli altri campi sono vuoti/nulli.
    """
    bc_pressure_zero = [
        name for name, bc_type in bc_config.items()
        if bc_type == "pressure_zero"
    ]

    bc = (
        bc_pressure_zero,   # bordi con p = 0 (Dirichlet)
        [],                 # pressione costante (non usato in modale)
        0.0,                # valore pressione costante
        [],                 # impedenza (non usato in modale)
        0.0 + 0j,           # valore impedenza
        [],                 # velocita' normale (non usato in modale)
        0.0,                # valore velocita' normale
        None,               # sorgente puntuale (non usato in modale)
        0.0,                # pressione sorgente puntuale
    )
    return bc


bc = build_bc_from_config(bc_config)
print("Formato bc creato correttamente.")

---
## Passo 5 — Prepara il sistema acustico (matrici FEM)

Questo è il cuore del metodo FEM.

Il software costruisce due grandi matrici:

- **Matrice di rigidezza acustica K**: contiene informazioni sulla geometria e sulla velocità del suono. Governa come si distribuisce la pressione nello spazio.
- **Matrice di massa acustica M**: contiene informazioni sulla densità del fluido. Governa l'inerzia del sistema (quanto "pesa" la risposta nel tempo / in frequenza).

Le condizioni al contorno di Dirichlet (`pressure_zero`) vengono applicate **riducendo** le matrici: i gradi di libertà con pressione imposta vengono eliminati dal sistema. Il risultato è il **sistema ridotto** `K_red` e `M_red`, che è quello che verrà risolto.

> **Analogia meccanica:** In meccanica strutturale, K è la rigidezza di una molla e M è la massa. Il problema modale `K φ = λ M φ` è esattamente lo stesso, con φ che è la forma del modo e λ = ω² il quadrato della pulsazione.

In [ ]:
system = pycafe.prepare_acoustic_system(
    nodes=nodes,
    elements=elements,
    boundaries=boundaries,
    rho=fluid["rho"],
    c0=fluid["c0"],
    bc=bc,
    debug=False,
)

K_red = system["K_red"]
M_red = system["M_red"]

print(f"Sistema preparato.")
print(f"Dimensione sistema ridotto: {K_red.shape[0]} gradi di liberta'")

---
## Passo 6 — Scegli il numero di modi e lancia l'analisi

L'analisi modale risolve il **problema agli autovalori**:

$$\mathbf{K}_{red}\, \boldsymbol{\varphi} = \lambda\, \mathbf{M}_{red}\, \boldsymbol{\varphi}$$

dove:
- $\lambda = \omega^2 = (2\pi f)^2$ è l'**autovalore** (quadrato della pulsazione)
- $\boldsymbol{\varphi}$ è l'**autovettore** (forma del modo, distribuzione spaziale di pressione)
- $f$ è la **frequenza propria** [Hz]

Il solver restituisce i `num_modes` autovalori più bassi (cioè le frequenze più basse), che sono i modi fisicamente più rilevanti (dominano la risposta acustica).

> **Quanti modi calcolare?** Di solito bastano i primi 5-10 per capire il comportamento del sistema. Più modi calcoli, più tempo richiede (anche se con mesh piccole il calcolo è rapidissimo).

In [ ]:
# ================================================================
# SCEGLI IL NUMERO DI MODI DA CALCOLARE
# (suggerito: tra 4 e 12 per iniziare)
# ================================================================
NUM_MODI = 6

# Risolve il problema agli autovalori
freqs, modes_red = solve_modal_acoustic_reduced(
    K_red,
    M_red,
    num_modes=NUM_MODI,
)

col_lam = "Lunghezza d'onda [m]"
print("\n=== FREQUENZE PROPRIE ===")
print(f"{'Modo':>6}  {'Frequenza [Hz]':>16}  {col_lam:>22}")
print("-" * 50)
for i, f in enumerate(freqs):
    lam = fluid["c0"] / f  # lunghezza d'onda = c0 / f
    print(f"  {i+1:>4}  {f:>16.3f}  {lam:>22.4f}")


---
## Passo 7 — Visualizza i modi propri

Ogni **modo proprio** è una distribuzione spaziale di pressione acustica.

Nella visualizzazione:
- **Rosso** → zona a pressione positiva (sovrapressione)
- **Blu** → zona a pressione negativa (depressione)
- **Bianco/grigio** → zona a pressione quasi nulla (nodo di pressione)

Le ampiezze sono **normalizzate** al massimo: quello che vedi è la *forma* del modo, non i valori assoluti di pressione.

> **Tip:** Osserva quanti "lobi" (regioni colorate) ci sono in x e in y: il modo (m,n) ha `m` semicicli nella direzione x e `n` nella direzione y.

In [ ]:
# ================================================================
# VISUALIZZA TUTTI I MODI IN UNA GRIGLIA
# (non modificare questa cella)
# ================================================================

n_cols = 3
n_rows = int(np.ceil(NUM_MODI / n_cols))

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(5 * n_cols, 4 * n_rows),
    squeeze=False
)

for i in range(NUM_MODI):
    ax = axes[i // n_cols][i % n_cols]

    # Espande il modo dal sistema ridotto al dominio completo
    mode_full = expand_mode_to_full(
        modes_red[:, i],
        system["idx_free"],
        system["p0_nodes"],
        nodes.shape[0]
    )

    # Normalizza
    mode_plot = np.real(mode_full)
    max_val = np.max(np.abs(mode_plot))
    if max_val > 0:
        mode_plot /= max_val

    sc = ax.scatter(
        nodes[:, 0], nodes[:, 1],
        c=mode_plot,
        cmap="seismic",
        vmin=-1, vmax=1,
        s=15
    )
    plt.colorbar(sc, ax=ax, label="Ampiezza norm.")
    ax.set_aspect("equal")
    ax.set_title(f"Modo {i+1} — f = {freqs[i]:.2f} Hz", fontsize=11)
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.grid(True, linewidth=0.3)

# Nasconde i subplot vuoti (se NUM_MODI non e' multiplo di n_cols)
for j in range(NUM_MODI, n_rows * n_cols):
    axes[j // n_cols][j % n_cols].set_visible(False)

fig.suptitle(
    f"Modi propri acustici — Fluido: {fluid['name']} — CC: {dict(bc_config)}",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.show()

---
## Passo 8 (opzionale) — Visualizza un singolo modo in dettaglio

Se vuoi esaminare un modo specifico in modo più dettagliato, modifica `MODO_DA_VEDERE` con il numero del modo (1 = primo modo, 2 = secondo, ecc.).

In [ ]:
# ================================================================
# VISUALIZZA UN SINGOLO MODO
# Cambia il numero qui sotto (1 = primo modo, 2 = secondo, ...)
# ================================================================
MODO_DA_VEDERE = 1

idx = MODO_DA_VEDERE - 1  # gli indici interni partono da 0

if idx < 0 or idx >= NUM_MODI:
    print(f"Numero modo non valido. Scegli tra 1 e {NUM_MODI}.")
else:
    mode_full = expand_mode_to_full(
        modes_red[:, idx],
        system["idx_free"],
        system["p0_nodes"],
        nodes.shape[0]
    )

    mode_plot = np.real(mode_full)
    max_val = np.max(np.abs(mode_plot))
    if max_val > 0:
        mode_plot /= max_val

    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(
        nodes[:, 0], nodes[:, 1],
        c=mode_plot,
        cmap="seismic",
        vmin=-1, vmax=1,
        s=25
    )
    plt.colorbar(sc, ax=ax, label="Ampiezza di pressione normalizzata [-]")
    ax.set_aspect("equal")
    ax.set_title(
        f"Modo {MODO_DA_VEDERE} — f = {freqs[idx]:.3f} Hz\n"
        f"Lunghezza d'onda: {fluid['c0']/freqs[idx]:.4f} m",
        fontsize=13
    )
    ax.set_xlabel("x [m]", fontsize=12)
    ax.set_ylabel("y [m]", fontsize=12)
    ax.grid(True)
    plt.tight_layout()
    plt.show()

    print(f"\nFrequenza:         {freqs[idx]:.3f} Hz")
    print(f"Pulsazione omega:  {2*np.pi*freqs[idx]:.3f} rad/s")
    print(f"Lunghezza d'onda:  {fluid['c0']/freqs[idx]:.4f} m")

---
## Esperimenti guidati

Prova a modificare la configurazione e osserva come cambiano i risultati. Qui sotto trovi alcune idee.

---

### Esperimento 1 — Stanza chiusa vs stanza aperta

**Configurazione A** (tutti hard_wall, stanza chiusa):
```python
bc_config = {
    "bottom": "hard_wall",
    "top":    "hard_wall",
    "left":   "hard_wall",
    "right":  "hard_wall",
}
```

**Configurazione B** (tutti pressure_zero, stanza aperta):
```python
bc_config = {
    "bottom": "pressure_zero",
    "top":    "pressure_zero",
    "left":   "pressure_zero",
    "right":  "pressure_zero",
}
```

**Domanda:** Le frequenze proprie sono le stesse? Come cambia la forma dei modi?

---

### Esperimento 2 — Condizione mista (un lato aperto)

```python
bc_config = {
    "bottom": "hard_wall",
    "top":    "pressure_zero",   # solo il tetto e' aperto
    "left":   "hard_wall",
    "right":  "hard_wall",
}
```

Questa configurazione è simile a un tubo chiuso su un'estremità e aperto sull'altra.
I modi dovrebbero avere frequenze a: `f = c0 / (4L) * (2n-1)` per la direzione y.

---

### Esperimento 3 — Cambia fluido

Imposta `FLUIDO = "water"` nel Passo 2 e riesegui tutto.

**Domanda:** Di quanto cambiano le frequenze proprie? Il rapporto è coerente con il rapporto c_acqua/c_aria ≈ 4.3?

---

### Esperimento 4 — Verifica analitica

Per una stanza rettangolare con **tutte le pareti rigide**, la formula analitica per le frequenze proprie è:

$$f_{m,n} = \frac{c_0}{2} \sqrt{\left(\frac{m}{L_x}\right)^2 + \left(\frac{n}{L_y}\right)^2}$$

dove `m, n = 0, 1, 2, ...` (ma non entrambi zero).

Usa la cella qui sotto per calcolare le frequenze analitiche e confrontarle con i risultati del FEM.

In [ ]:
# ================================================================
# VERIFICA ANALITICA (solo per stanza rettangolare con hard wall)
# Modifica Lx e Ly con le dimensioni reali della tua mesh.
# ================================================================

Lx = nodes[:, 0].max() - nodes[:, 0].min()  # dimensione in x
Ly = nodes[:, 1].max() - nodes[:, 1].min()  # dimensione in y

print(f"Dimensioni del dominio: Lx = {Lx:.4f} m, Ly = {Ly:.4f} m")
print(f"Velocita' del suono   : c0 = {fluid['c0']} m/s\n")

# Calcola i primi modi analitici
freqs_analytic = []
for m in range(5):
    for n in range(5):
        if m == 0 and n == 0:
            continue  # modo triviale (f=0)
        f_mn = (fluid["c0"] / 2) * np.sqrt((m / Lx)**2 + (n / Ly)**2)
        freqs_analytic.append((f_mn, m, n))

freqs_analytic.sort()

print(f"{'Modo (m,n)':>12}  {'f analitica [Hz]':>18}  {'f FEM [Hz]':>14}  {'Errore [%]':>12}")
print("-" * 62)
for i, (f_an, m, n) in enumerate(freqs_analytic[:NUM_MODI]):
    if i < len(freqs):
        f_fem = freqs[i]
        err = abs(f_fem - f_an) / f_an * 100
        print(f"({m},{n}){' ':>8}  {f_an:>18.3f}  {f_fem:>14.3f}  {err:>11.2f}%")
    else:
        print(f"({m},{n}){' ':>8}  {f_an:>18.3f}  {'---':>14}")

print("\n(I valori analitici sono validi solo per tutti-hard-wall e mesh rettangolare)")

---

## Riepilogo del flusso di lavoro

```
1. Scegli fluido   →  pycafe.load_fluid()
2. Carica mesh     →  pycafe.load_mesh()
3. Configura CC    →  dizionario bc_config  →  build_bc_from_config()
4. Prepara sistema →  pycafe.prepare_acoustic_system()
5. Risolvi         →  solve_modal_acoustic_reduced(K_red, M_red, num_modes)
6. Visualizza      →  expand_mode_to_full() + matplotlib
```

Per passare all'**analisi in frequenza** (risposta forzata), guarda il notebook `Example1.ipynb` e seleziona `analysis_type = "direct"` nel driver interattivo.

---
*Notebook creato con pyCAFE — Computational Acoustics Finite Elements*